# CiteCheck Quickstart

**Agentic-RAG for verifiable US case-law citations.**

This notebook is an end-to-end *demo* of the CiteCheck pipeline using fully
mocked components. Run it after `pip install -e .` to walk through every
moving part of the system without needing a GPU, network access, the full
Caselaw Access Project corpus, the CourtListener API, or an LLM endpoint.

## What this notebook does

1. Loads a tiny 10-example smoke eval set spanning verified, non-supporting,
   fabricated, and malformed citations.
2. Demonstrates the Bluebook citation grammar.
3. Wires up an in-notebook mock CourtListener client + in-memory retriever +
   canned LLM generator.
4. Runs all three Phase-2 systems on the same three questions:
   - `VanillaBaseline` (parametric only, no retrieval)
   - `NaiveRAGBaseline` (retrieve top-k, stuff, generate)
   - `VerifyLoop` (CiteCheck's agentic verify-then-emit loop)
5. Computes the 7 evaluation metrics with `aggregate_metrics` and prints the
   resulting `EvaluationReport`.

## What this notebook is NOT

**The metrics produced here are not meaningful performance numbers.** Every
component is mocked: the generator returns canned strings, the retriever does
naive keyword matching, the CourtListener client is a hard-coded dict, and the
NLI judge is monkeypatched to a deterministic stub. The goal is to let you
*see how the pieces fit together*, not to measure system quality.

For the real evaluation pipeline see
`docs/superpowers/plans/2026-05-24-phd-prep-phase-2-build.md`.

## Estimated runtime

End-to-end: under 30 seconds on a laptop CPU (most of which is package
imports). A new reader should plan ~20-30 minutes to step through cells and
read explanations.

## Setup

Imports + logging. Everything from `citecheck.*` is plain Python — no torch
or transformers required for the imports themselves (the heavy deps are
lazy-loaded inside `CitationResolver._score_entailment` which we monkeypatch
out below).

In [ ]:
from __future__ import annotations

import logging
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

# CiteCheck public surface
from citecheck.agent import (
    AnswerWithCitations as AgentAnswer,
    BluebookGrammar,
    CitationResolver,
    CitationStatus as AgentCitationStatus,
    VerificationResult as AgentVerificationResult,
    VerifyLoop,
)
from citecheck.baselines import NaiveRAGBaseline, VanillaBaseline
from citecheck.data import CiteCheckExample, load_eval_set
from citecheck.data.cl_client import CLOpinion
from citecheck.eval import aggregate_metrics
from citecheck.eval.types import (
    AnswerWithCitations,
    CitationStatus,
    VerificationResult,
)
from citecheck.retrieval import RetrievalResult

logging.basicConfig(
    level=logging.WARNING,
    format="%(name)s %(levelname)s: %(message)s",
)
logging.getLogger("citecheck").setLevel(logging.INFO)
print("citecheck imports OK")

## Locate the fixtures

The notebook lives at `citecheck/src/citecheck/notebooks/quickstart.ipynb`
and the fixtures at `citecheck/tests/fixtures/`. Resolve those paths relative
to the project root regardless of where Jupyter was launched from.

In [ ]:
import citecheck

# citecheck/__init__.py lives at .../citecheck/src/citecheck/__init__.py
_PKG_ROOT = Path(citecheck.__file__).resolve().parent          # .../src/citecheck
_PROJECT_ROOT = _PKG_ROOT.parent.parent                         # .../citecheck
FIXTURES = _PROJECT_ROOT / "tests" / "fixtures"

SMOKE_EVAL = FIXTURES / "smoke_eval.jsonl"
SMOKE_CORPUS = FIXTURES / "smoke_corpus.jsonl"

assert SMOKE_EVAL.exists(), f"expected fixture at {SMOKE_EVAL}"
assert SMOKE_CORPUS.exists(), f"expected fixture at {SMOKE_CORPUS}"
print("smoke_eval:  ", SMOKE_EVAL)
print("smoke_corpus:", SMOKE_CORPUS)

## Section 1: Load the smoke eval set

`load_eval_set` parses the JSONL into a list of `CiteCheckExample` dataclasses.
Each example carries the question, a list of `GoldCitation` objects, the
jurisdiction, and free-form metadata. The smoke set spans four axes encoded
in `metadata['axis']`:

* `verified-supported` — citation resolves AND opinion entails the claim
* `verified-non-supporting` — citation resolves but the opinion is off-topic
* `unresolvable-fabricated` — Bluebook-shaped but the case does not exist
* `malformed` — does not parse as a Bluebook citation at all

In [ ]:
from collections import Counter

EVAL_SET: list[CiteCheckExample] = load_eval_set(SMOKE_EVAL)
print(f"Loaded {len(EVAL_SET)} smoke examples")
print("Axis distribution:", dict(Counter(ex.metadata.get("axis", "?") for ex in EVAL_SET)))

for ex in EVAL_SET[:2]:
    print("\nid:        ", ex.id)
    print("question:  ", ex.question)
    print("jurisdiction:", ex.jurisdiction)
    print("axis:      ", ex.metadata.get("axis"))
    for gc in ex.gold_citations:
        print(f"  gold: {gc.citation}  (cl_opinion_id={gc.cl_opinion_id})")

## Section 2: Inspect the Bluebook grammar

`BluebookGrammar` builds a regex covering full-form case citations of the
shape `<Case Name>, <vol> <Reporter> <page> (<court> <year>)`. It is used as
(a) the *constrained-decoding* guard so the LLM cannot emit a syntactically
broken citation and (b) a cheap pre-filter before we hand spans to `eyecite`
for parsing.

We validate two real and two fake-looking strings. Note that *grammatically
valid* is **not** the same as *real*: `Doe v. Roe, 999 F.4th 1 (12th Cir.
9999)` parses cleanly even though the 12th Circuit does not exist. Catching
that is the job of the resolver in Section 4, not the grammar.

In [ ]:
grammar = BluebookGrammar()

samples = [
    ("Brown v. Board of Education, 347 U.S. 483 (1954)", "real SCOTUS"),
    ("Smith v. Jones, 412 F.3d 567 (9th Cir. 2005)",       "synthetic, real-looking"),
    ("Doe v. Roe, 999 F.4th 1 (12th Cir. 9999)",            "fabricated (12th Cir. does not exist)"),
    ("Phantom v. Specter, 222 Fake Rep. 333 (2099)",        "unknown reporter; should fail"),
    ("this is not a citation",                                "plain English; should fail"),
]
for text, label in samples:
    print(f"{str(grammar.validate(text)):5s}  {label:38s}  {text}")

print(f"\nRegex length: {len(grammar.bluebook_regex)} chars")
print("First 220 chars of regex:")
print(grammar.bluebook_regex[:220], "...")

## Section 3: Mock CourtListener client + a stub CitationResolver

In production `CitationResolver` does three things per citation:

1. Lift the Bluebook span via `eyecite.get_citations`.
2. Hit the CourtListener REST v4 API for canonical resolution.
3. Score entailment of the asserted claim against the opinion body with an
   NLI judge (DeBERTa-v3-large MNLI/FEVER/ANLI by default).

All three are expensive (eyecite is fast but optional, CL is rate-limited,
the NLI judge needs a GPU for tolerable latency). For the smoke notebook we
replace each with a deterministic stub.

### MockCourtListenerClient

A canned dict of `citation_str -> CLOpinion`. Returns `None` for anything not
in the dict — that is exactly how the real client signals "no such case".

In [ ]:
class MockCourtListenerClient:
    """Drop-in replacement for citecheck.data.CourtListenerClient.

    Knows about exactly the citations that appear in smoke_corpus; anything
    else (including the deliberately fabricated cites in smoke_eval) resolves
    to ``None``.
    """

    def __init__(self, known: dict[str, CLOpinion]):
        self._known = known
        self.resolve_calls: list[str] = []

    def resolve_citation(self, citation_str: str) -> CLOpinion | None:
        self.resolve_calls.append(citation_str)
        for needle, opinion in self._known.items():
            if needle.lower() in citation_str.lower():
                return opinion
        return None

    def get_opinion(self, opinion_id: int) -> CLOpinion | None:
        for op in self._known.values():
            if op.id == opinion_id:
                return op
        return None

    def close(self) -> None:
        pass


def _opinion(
    opinion_id: int, case: str, court: str, jur: str, year: int,
    cite: str, body: str,
) -> CLOpinion:
    return CLOpinion(
        id=opinion_id, case_name=case, court=court, court_jurisdiction=jur,
        year=year, citation_strings=[cite], body_text=body, url="",
    )


_KNOWN_OPINIONS: dict[str, CLOpinion] = {
    "412 F.3d 567": _opinion(
        100001, "Smith v. Jones", "Ninth Circuit", "F-9", 2005,
        "412 F.3d 567",
        "A fixture-supplier is liable when the design defect was foreseeable.",
    ),
    "477 U.S. 317": _opinion(
        100002, "Celotex Corp. v. Catrett", "Supreme Court", "SCOTUS", 1986,
        "477 U.S. 317",
        "A party seeking summary judgment may demonstrate the absence of evidence.",
    ),
    "347 U.S. 483": _opinion(
        100003, "Brown v. Board of Education", "Supreme Court", "SCOTUS", 1954,
        "347 U.S. 483",
        "Separate educational facilities are inherently unequal.",
    ),
    "304 U.S. 64": _opinion(
        100006, "Erie R. Co. v. Tompkins", "Supreme Court", "SCOTUS", 1938,
        "304 U.S. 64",
        "Federal courts sitting in diversity apply state substantive law.",
    ),
    "585 U.S. 296": _opinion(
        100007, "Carpenter v. United States", "Supreme Court", "SCOTUS", 2018,
        "585 U.S. 296",
        "Acquisition of cell-site location records is a Fourth Amendment search.",
    ),
}

mock_cl = MockCourtListenerClient(_KNOWN_OPINIONS)

# Smoke-test the mock against one real and one fake string.
for cite in [
    "Smith v. Jones, 412 F.3d 567 (9th Cir. 2005)",
    "Doe v. Roe, 999 F.4th 1 (12th Cir. 9999)",
]:
    op = mock_cl.resolve_citation(cite)
    print(f"{cite:55s} -> {'HIT  ' + op.case_name if op else 'MISS (unresolvable)'}")

### Stub the NLI judge

`CitationResolver._score_entailment` lazy-loads DeBERTa. We monkeypatch it on
the instance so the model never loads. The stub returns a deterministic score
based on simple keyword overlap so that downstream `VERIFIED` /
`NON_SUPPORTING` verdicts behave the way the real judge roughly would on this
tiny corpus.

We also bypass `parse_bluebook` (which depends on `eyecite`) and route
extraction through the grammar's regex. This is purely a demo shortcut; the
real pipeline always uses eyecite for authoritative parsing.

In [ ]:
from citecheck.agent.citation_resolver import ParsedCitation

# A tight Bluebook regex that keeps the (Court Year) parenthetical attached.
# The production grammar makes the parenthetical optional, which lets find_all
# emit weird spans; for the demo we want clean strings.
_DEMO_CITE_RE = re.compile(
    r"[A-Z][A-Za-z.&'\-]*(?:\s+[A-Z][A-Za-z.&'\-]*)*"
    r"\s+v\.?\s+"
    r"[A-Z][A-Za-z.&'\-]*(?:\s+[A-Z][A-Za-z.&'\-]*)*"
    r",\s+\d{1,4}\s+[A-Z][A-Za-z0-9.\s]*?\s+\d{1,6}"
    r"(?:\s+\([^)]+\))?"
)


def _demo_extract(text: str) -> list[str]:
    """Pull citation-shaped substrings from text without invoking eyecite."""
    return [m.group(0).strip() for m in _DEMO_CITE_RE.finditer(text)]


def _demo_parse_bluebook(self, text: str) -> list[ParsedCitation]:  # noqa: ARG001
    """Stand-in for CitationResolver.parse_bluebook that uses the demo regex."""
    return [ParsedCitation(raw_text=s) for s in _demo_extract(text)]


def _demo_score_entailment(self, claim: str, body: str) -> float:  # noqa: ARG001
    """Deterministic stub for the NLI judge.

    Returns 0.9 when 3+ tokens of the body appear in the claim, 0.2 otherwise.
    Real CiteCheck uses DeBERTa-v3-large MNLI/FEVER/ANLI here.
    """
    claim_tokens = {t.lower() for t in re.findall(r"\w+", claim) if len(t) >= 4}
    body_tokens = {t.lower() for t in re.findall(r"\w+", body) if len(t) >= 4}
    overlap = len(claim_tokens & body_tokens)
    return 0.9 if overlap >= 3 else 0.2


resolver = CitationResolver(cl_client=mock_cl)
# Monkeypatch the instance methods so no eyecite + no DeBERTa is loaded.
resolver.parse_bluebook = _demo_parse_bluebook.__get__(resolver, CitationResolver)
resolver._score_entailment = _demo_score_entailment.__get__(resolver, CitationResolver)

# Quick verify of the wiring on one real and one fake citation.
for cite, claim in [
    (
        "Smith v. Jones, 412 F.3d 567 (9th Cir. 2005)",
        "A fixture-supplier is liable when the design defect was foreseeable.",
    ),
    (
        "Doe v. Roe, 999 F.4th 1 (12th Cir. 9999)",
        "An at-will employee has a wrongful-discharge claim.",
    ),
]:
    v = resolver.verify(cite, asserted_claim=claim)
    print(f"  {cite}")
    print(f"    status: {v.status.value}  notes: {v.notes}")

## Section 4: Mock retrieval

In production CiteCheck uses `HybridRetriever` (BM25 via Pyserini + a dense
bi-encoder + RRF fusion) over the indexed CAP corpus. Building that index
needs Java, ~30 GB of opinion text, and a GPU for the dense pass.

For the smoke notebook we wrap the 20 corpus passages in an in-memory
retriever that does naive case-insensitive token overlap scoring. It exposes
the same `.search(query, top_k)` contract that `VerifyLoop` and
`NaiveRAGBaseline` expect, returning a list of `RetrievalResult`.

In [ ]:
import json


@dataclass
class _CorpusEntry:
    doc_id: str
    contents: str
    tokens: set[str] = field(default_factory=set)


def _tokenize(text: str) -> set[str]:
    return {t.lower() for t in re.findall(r"[a-zA-Z]{3,}", text)}


def _load_corpus(path: Path) -> list[_CorpusEntry]:
    out: list[_CorpusEntry] = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        d = json.loads(line)
        out.append(_CorpusEntry(doc_id=d["id"], contents=d["contents"],
                                tokens=_tokenize(d["contents"])))
    return out


class InMemoryRetriever:
    """Tiny stand-in for HybridRetriever.

    Ranking is naive token overlap. Good enough for a 20-doc smoke corpus;
    completely inadequate for the real CAP-scale eval, which is exactly why we
    use BM25 + dense + RRF in production.
    """

    def __init__(self, corpus: list[_CorpusEntry]):
        self.corpus = corpus

    def search(self, query: str, top_k: int = 5) -> list[RetrievalResult]:
        q = _tokenize(query)
        scored = [
            (len(q & e.tokens), e) for e in self.corpus
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        results: list[RetrievalResult] = []
        for rank, (score, entry) in enumerate(scored[:top_k], start=1):
            if score == 0:
                break
            results.append(RetrievalResult(
                doc_id=entry.doc_id,
                score=float(score),
                rank=rank,
                passage=entry.contents,
                metadata={"source": "smoke_corpus"},
            ))
        return results


corpus = _load_corpus(SMOKE_CORPUS)
retriever = InMemoryRetriever(corpus)
print(f"Loaded {len(corpus)} corpus passages")

# Demo: search for the first three smoke questions, show the top 3 hits each.
for ex in EVAL_SET[:3]:
    hits = retriever.search(ex.question, top_k=3)
    print(f"\nQ: {ex.question}")
    for h in hits:
        print(f"  [{h.rank}] {h.doc_id}  score={h.score:.0f}  {h.passage[:80]}...")

## Section 5: Mock the LLM generator

Baselines and `VerifyLoop` take any `generator: Callable[[str], str]`. We
build one that returns a canned answer per question, with a deliberate mix:
some answers cite verifiable cases, some cite fabrications, some cite
non-supporting opinions. This lets the verify loop *do something* on each
iteration.

The generator picks its answer based on substring match in the prompt — that
is the same hook the real LLM uses (the question is always at the end of the
prompt under `### Question`).

In [ ]:
# Each key is a substring of a smoke question; first match wins.
_CANNED: list[tuple[str, str]] = [
    (
        "fixture-supplier",
        "Yes. A fixture-supplier is liable for foreseeable design defects.\n"
        "See Smith v. Jones, 412 F.3d 567 (9th Cir. 2005).",
    ),
    (
        "summary judgment",
        "The non-moving party must produce evidence of a genuine dispute.\n"
        "See Celotex Corp. v. Catrett, 477 U.S. 317 (1986).",
    ),
    (
        "Erie",
        "Federal courts in diversity apply state substantive law.\n"
        "See Erie R. Co. v. Tompkins, 304 U.S. 64 (1938).",
    ),
    (
        "Brown v. Board",
        "Brown overruled Plessy with respect to public-school segregation.\n"
        "See Brown v. Board of Education, 347 U.S. 483 (1954).",
    ),
    (
        "punitive damages",
        "Punitive damages for ordinary contract breach are generally unavailable.\n"
        "See Celotex Corp. v. Catrett, 477 U.S. 317 (1986).",
    ),
    (
        "at-will employee in Delaware",
        "Some states recognize a public-policy exception.\n"
        "See Doe v. Roe, 999 F.4th 1 (12th Cir. 9999).",
    ),
    (
        "piercing the corporate veil",
        "Texas applies a sham-to-perpetrate-fraud standard.\n"
        "See Acme v. Beta, 888 F.5th 1 (99th Cir. 8888).",
    ),
    (
        "cell-site",
        "The acquisition of cell-site location records is a Fourth Amendment search.\n"
        "See Carpenter v. United States, 585 U.S. 296 (2018).",
    ),
    (
        "qualified immunity",
        "Government officials enjoy qualified immunity.\n"
        "See Phantom v. Specter, 222 Fake Rep. 333 (2099).",
    ),
    (
        "rule against perpetuities",
        "An interest must vest within 21 years after a life in being.\n"
        "see citation but no proper Bluebook form",
    ),
]


class MockGenerator:
    """Callable that returns a canned answer for any prompt mentioning a known question."""

    model_name = "mock-generator-v0"

    def __init__(self) -> None:
        self.calls: list[str] = []

    def __call__(self, prompt: str) -> str:
        self.calls.append(prompt)
        for needle, answer in _CANNED:
            if needle.lower() in prompt.lower():
                return answer
        return "I do not have a verified citation for that question."


generator = MockGenerator()
print("Demo generator call:")
print(generator("### Question\nWhat is the Erie doctrine?\n### Answer\n"))

## Section 6: VanillaBaseline (parametric only, no retrieval)

`VanillaBaseline` hits the LLM with the question and a strict citation-format
instruction. There is no retrieval and no verification feedback loop — the
resolver is only consulted *post-hoc* to populate the `citations` field so
the eval harness can score the same way it does for RAG systems.

It returns an `agent.AnswerWithCitations` (text + list of
`(citation_str, VerificationResult)` tuples). The eval layer's
`AnswerWithCitations.from_agent_answer` adapter bridges that shape to the
eval-layer shape consumed by the metrics.

In [ ]:
vanilla = VanillaBaseline(generator=generator, resolver=resolver)

vanilla_demo_ids = ["smoke001", "smoke006", "smoke009"]  # verified, fabricated, fabricated
vanilla_results: list[tuple[CiteCheckExample, AgentAnswer]] = []

for ex in [e for e in EVAL_SET if e.id in vanilla_demo_ids]:
    ans = vanilla.answer(ex.question)
    vanilla_results.append((ex, ans))
    print(f"\n=== {ex.id} ({ex.metadata.get('axis')})")
    print(f"Q: {ex.question}")
    print(f"A: {ans.text}")
    print(f"Iterations used: {ans.iterations_used}")
    for cite_str, vr in ans.citations:
        print(f"  cite: {cite_str}  status: {vr.status.value}")

## Section 7: NaiveRAGBaseline

Same mock generator + the in-memory retriever. The baseline now retrieves the
top-k passages, stuffs them into the prompt under `### Context`, and asks the
generator to answer. Verification is still post-hoc (no feedback loop).

In [ ]:
naive = NaiveRAGBaseline(
    generator=generator, retriever=retriever, resolver=resolver, top_k=3,
)

naive_results: list[tuple[CiteCheckExample, AgentAnswer]] = []
for ex in [e for e in EVAL_SET if e.id in vanilla_demo_ids]:
    ans = naive.answer(ex.question)
    naive_results.append((ex, ans))
    print(f"\n=== {ex.id}")
    print(f"A: {ans.text}")
    print(f"Retrieved docs: {ans.retrieved_doc_ids}")
    for cite_str, vr in ans.citations:
        print(f"  cite: {cite_str}  status: {vr.status.value}")

### Peek at the prompt NaiveRAG actually sent

Helpful for understanding what the generator sees. The mock generator keeps
every prompt it has been called with in `generator.calls`.

In [ ]:
last_prompt = generator.calls[-1]
print("Last prompt sent by NaiveRAGBaseline:\n")
print(last_prompt)

## Section 8: The full VerifyLoop

`VerifyLoop` is CiteCheck's main contribution: after each generation it parses
the citations, verifies each one, and if any failed (`UNRESOLVABLE` or
`NON_SUPPORTING`) it re-prompts the generator with verification feedback. The
loop stops when (a) all citations VERIFY, (b) the answer contains no
citations, or (c) `max_iterations` is exhausted.

We disable constrained decoding for the demo (`use_constrained_decoding=False`)
because the mock generator doesn't honour grammar constraints anyway. In
production this flag wires the Bluebook regex into Outlines / `transformers-cfg`
so the LLM literally cannot emit a syntactically broken citation.

In [ ]:
loop = VerifyLoop(
    generator=generator,
    retriever=retriever,
    reranker=None,
    resolver=resolver,
    max_iterations=2,
    use_constrained_decoding=False,
    top_k=3,
)

verify_results: list[tuple[CiteCheckExample, AgentAnswer]] = []
for ex in [e for e in EVAL_SET if e.id in vanilla_demo_ids]:
    ans = loop.answer(ex.question)
    verify_results.append((ex, ans))
    print(f"\n=== {ex.id} ({ex.metadata.get('axis')})")
    print(f"A: {ans.text}")
    print(f"Iterations used: {ans.iterations_used}")
    for cite_str, vr in ans.citations:
        print(f"  cite: {cite_str}  status: {vr.status.value}")

Notice what happened on `smoke006` (the at-will-employee question with the
fabricated `Doe v. Roe` citation). The loop's first generation returned the
canned answer with the fabricated cite. The resolver flagged it as
`UNRESOLVABLE`, the loop re-prompted with feedback, and the canned generator
(having no better answer) returned the same string. After two iterations the
budget was exhausted and the loop returned the un-VERIFIED answer.

With a real LLM the second pass would either substitute a real cite or drop
the claim. The mock generator's repetition is *itself* a useful demo — it
shows the loop terminates cleanly even when the generator is stuck.

## Section 9: Compute the 7 metrics

Run the `VanillaBaseline` over all 10 smoke examples and aggregate. The
metrics expect the *eval-layer* `AnswerWithCitations` shape with a
`question_id` and `verification_results: list[VerificationResult]`. We use
`AnswerWithCitations.from_agent_answer` to bridge.

Reminder: these numbers are an artifact of the mocks. They are useful only as
a sanity check that the pipeline runs end-to-end and the report serializes.

In [ ]:
predictions: list[AnswerWithCitations] = []
for ex in EVAL_SET:
    agent_ans = vanilla.answer(ex.question)
    pred = AnswerWithCitations.from_agent_answer(
        agent_ans,
        question_id=ex.id,
        latency_ms=12.3,        # mocked; real runs measure this
        tokens_used=128,         # mocked; real runs report from the LLM
    )
    predictions.append(pred)

print(f"Built {len(predictions)} predictions")
for p in predictions[:3]:
    print(f"  {p.question_id}: cites={len(p.citations)} "
          f"verifications={[v.status.value for v in p.verification_results]}")

In [ ]:
report = aggregate_metrics(
    predictions=predictions,
    gold=EVAL_SET,
    baseline_name="VanillaBaseline (mocked)",
    model_name=generator.model_name,
)

print("Headline metrics:")
print(f"  sample_size:              {report.sample_size}")
print(f"  resolution_rate:          {report.resolution_rate:.3f}")
print(f"  fabrication_rate:         {report.fabrication_rate:.3f}")
print(f"  jurisdictional_validity:  {report.jurisdictional_validity:.3f}")
print(f"  citation_support_f1:      {report.citation_support_f1}")
print(f"  latency_ms:               {report.latency_ms}")
print(f"  tokens:                   {report.tokens}")
print("\nPer-jurisdiction breakdown:")
for jur, stats in report.per_jurisdiction.items():
    print(f"  {jur:14s}  {stats}")

In [ ]:
# Full JSON report (this is what EvaluationRunner would persist to disk).
print(report.to_json())

## Section 10: What this notebook did NOT show

For an honest end-to-end Phase 2 run you would replace every mock in this
notebook with the production component. Concretely:

1. **Corpus.** Download the Caselaw Access Project bulk dump
   (`citecheck.data.download_cap_bulk`), parse to parquet, and chunk to ~512
   tokens. Expect ~30 GB of opinion text.
2. **Indexes.** Build the BM25 index with Pyserini's Lucene indexer and the
   dense FAISS index with `BAAI/bge-base-en-v1.5`. The dense pass needs a GPU.
3. **Reranker.** Train the dual-head cross-encoder on hard negatives via
   `citecheck.reranker.train_reranker`. ~30 minutes on a single A100.
4. **LLM generator.** Wire `transformers` / Outlines / OpenRouter to a real
   instruction-tuned model (Llama-3.1-8B by default; see
   `citecheck.config.MODELS`).
5. **CourtListener.** Drop `MockCourtListenerClient` in favour of the real
   `citecheck.data.CourtListenerClient`. Set `CL_API_KEY` in `.env`.
6. **NLI judge.** Remove the `_demo_score_entailment` monkeypatch. The first
   call lazy-loads DeBERTa-v3-large MNLI/FEVER/ANLI (~1.5 GB on GPU).
7. **Eval set.** Replace `smoke_eval.jsonl` with the real CiteCheck eval set
   built by `seed_from_charlotin` -> `llm_prelabel` -> `human_verify_subset`.
8. **Eval harness.** Use `citecheck.eval.EvaluationRunner` instead of the
   hand-rolled loop in Section 9 — the runner streams a per-question JSONL
   trace, captures real latency, and writes the report JSON to disk.
9. **Human audit.** Run the 100-item utility audit and pass
   `human_ratings={question_id: 1..5}` so `answer_utility` is non-NaN.

The full Phase 2 build plan, including milestones and acceptance criteria,
is at `docs/superpowers/plans/2026-05-24-phd-prep-phase-2-build.md`.

Where to go next:
- `citecheck/README.md` — repo overview
- `citecheck/Makefile` — `make test`, `make data`, `make index`, `make eval`
- `citecheck/docs/architecture.md` — module-level design notes
- The pytest suite at `citecheck/tests/` — every module has unit tests that
  use the same mock pattern as this notebook.